# Non-Linear Economy Estimation (ANN)

Please note that this notebook uses a venv which points to a base python version of **3.13**, some functionality may be limited if using an older version of python.

## All Imports

In [4]:
%pip install scikit-learn seaborn torch torchvision gymnasium stable-baselines3 --quiet

Note: you may need to restart the kernel to use updated packages.


In [5]:
%matplotlib widget

## Data From Source Package

In [6]:
%pip install -e ../ --quiet

Note: you may need to restart the kernel to use updated packages.


In [7]:
import autonomous_fed as afed

In [8]:
le_solver = afed.LinearEnvironmentSolver(fred_key="7ab121fb17773e187bb6508e83e411da")

# Data Check
print (le_solver.historical_data.head())
print (le_solver.historical_data.tail())

             pi         y     i
date                           
1987Q3  2.66122 -0.388640  6.84
1987Q4  2.91876  0.526730  6.92
1988Q1  3.06577  0.260731  6.66
1988Q2  3.35274  0.788853  7.16
1988Q3  3.80207  0.595530  7.98
             pi         y     i
date                           
2006Q2  3.35642  1.319357  4.91
2006Q3  3.13805  0.978513  5.25
2006Q4  2.66316  1.380249  5.25
2007Q1  2.91019  1.210046  5.26
2007Q2  2.72883  1.336951  5.25


## Model Architecture

### NARX Model

For our economy transition equations we have: ${y_t=\hat{f}^y(y_{t-1},y_{t-2},\pi_t,\pi_{t-1},\pi_{t-2},i_t,i_{t-1},i_{t-2})+\epsilon_t^y}$ and ${\pi_t=\hat{f}^\pi(y_t,y_{t-1},y_{t-2},\pi_{t-1},\pi_{t-2},i_t,i_{t-1},i_{t-2})+\epsilon_t^\pi}$

In this scenario our predictor function, ${\hat{f}}$ is an ANN (Artificial Neural Network) but it can be swapped with other nonlinear functions such as a sigmoid function or wavelet network. For our ANN predictor we have ${\hat{f}^m=b_0^m+\sum_{j=1}^h v_j^mG(\omega_j^{m'}s_t^m+b_j^m), m\in\{y,\pi}\}$

The Components of the ANN are as follows:
- ${m}$: ${\{y,\pi}\}$
- ${s_t^m}$: Input state vectors at time t.
- ${w_j^m}$: Weight vector for the j-th hidden neuron.
- ${b_j^m}$: Bias term for the current neuron.
- ${G(\cdot)}$: Activation function (nonlinear transform).
- ${v_j^m}$: Weight from hidden neuron ${j}$ to the output layer
- ${b_0^m}$: Bias at the output layer
- ${h}$: Number of hidden neurons.

As seen above the Neural Network type is a NARX Model with a single hidden layer and the activation function is the hyperbolic tangent therefore we define ${G(\cdot)}$ as follows: ${G(x)=tanh(x)=\frac{e^x-e^-x}{e^x+e^-x}}$

### Model Buildout

In [ ]:
# PyTorch NARX-style single-hidden-layer ANN (tanh) — “eq. (8)” replica
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# -----------------------------
# 1) Prepare supervised NARX data
# -----------------------------
def make_lag_matrix(df, target_col, lag_cols, p=4, holdout_frac=0.15):
    """
    df: DataFrame with time-ordered rows and columns including target and inputs
    target_col: e.g. 'y' or 'pi'
    lag_cols: list of columns to lag (endogenous + exogenous), e.g. ['y','pi','i']
    p: number of lags
    holdout_frac: last % of samples reserved for validation (like the paper)
    """
    # Build lagged design matrix
    X_parts = []
    for col in lag_cols:
        for L in range(1, p+1):
            X_parts.append(df[col].shift(L).rename(f"{col}_lag{L}"))
    X = pd.concat(X_parts, axis=1)
    y = df[target_col].copy()

    # Drop initial p rows with NaNs from lagging
    X, y = X.iloc[p:], y.iloc[p:]

    # Train/val split (last 15% is validation)
    n = len(X)
    n_val = int(np.ceil(n * holdout_frac))
    split = n - n_val

    X_train, X_val = X.iloc[:split], X.iloc[split:]
    y_train, y_val = y.iloc[:split], y.iloc[split:]

    # Standardize inputs (fit on train, apply to val)
    mu, sigma = X_train.mean(), X_train.std().replace(0, 1.0)
    X_train_z = (X_train - mu) / sigma
    X_val_z   = (X_val   - mu) / sigma

    # Tensors
    Xt = torch.tensor(X_train_z.values, dtype=torch.float32)
    Xv = torch.tensor(X_val_z.values,   dtype=torch.float32)
    yt = torch.tensor(y_train.values,   dtype=torch.float32).view(-1, 1)
    yv = torch.tensor(y_val.values,     dtype=torch.float32).view(-1, 1)
    return Xt, yt, Xv, yv, X.columns.tolist(), (mu, sigma)

# -----------------------------
# 2) Model = single hidden layer tanh, linear output
# -----------------------------
class Eq8Net(nn.Module):
    def __init__(self, in_dim, h):
        super().__init__()
        self.hidden = nn.Linear(in_dim, h)      # ω_j
        self.act    = nn.Tanh()                 # G(·) = tanh
        self.out    = nn.Linear(h, 1)           # ν_j and b0
    def forward(self, x):
        return self.out(self.act(self.hidden(x)))

# -----------------------------
# 3) Train with early stopping
# -----------------------------
def fit_once(Xt, yt, Xv, yv, h, seed, lr=1e-3, batch=128, max_epochs=2000, patience=40):
    torch.manual_seed(seed)
    model = Eq8Net(Xt.shape[1], h)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    train_loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch, shuffle=True)

    best_state, best_val, wait = None, float('inf'), 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
        # validation
        model.eval()
        with torch.no_grad():
            val = loss_fn(model(Xv), yv).item()
        if val + 1e-10 < best_val:
            best_val = val
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val

def fit_with_model_selection(Xt, yt, Xv, yv, h_grid=range(1, 11), restarts=30):
    best_model, best_h, best_val = None, None, float('inf')
    for h in h_grid:
        for seed in range(restarts):
            model, val = fit_once(Xt, yt, Xv, yv, h=h, seed=seed)
            if val < best_val:
                best_model, best_h, best_val = model, h, val
    return best_model, best_h, best_val

# -----------------------------
# 4) Example usage
# -----------------------------
# Suppose df has columns ['y','pi','i', ...] at quarterly frequency, sorted by time
# df = your_dataframe

# Choose which columns provide the state s_t^m (endogenous + exogenous)
# lag_cols = ['y', 'pi', 'i']   # adjust to match your linear economy state vector

# y-model
# Xt_y, yt_y, Xv_y, yv_y, cols_y, scaler_y = make_lag_matrix(df, target_col='y',  lag_cols=lag_cols, p=4, holdout_frac=0.15)
# model_y, h_y, val_y = fit_with_model_selection(Xt_y, yt_y, Xv_y, yv_y)

# pi-model
# Xt_p, yt_p, Xv_p, yv_p, cols_p, scaler_p = make_lag_matrix(df, target_col='pi', lag_cols=lag_cols, p=4, holdout_frac=0.15)
# model_p, h_p, val_p = fit_with_model_selection(Xt_p, yt_p, Xv_p, yv_p)

# print("Best h (y):", h_y, "Val MSE (y):", val_y)
# print("Best h (pi):", h_p, "Val MSE (pi):", val_p)

# -----------------------------
# 5) Dynamic (multi-step) forecasting like NARX
# -----------------------------
@torch.no_grad()
def dynamic_forecast(model, df_last, lag_cols, p, scaler, horizon, target_name, other_targets_updater=None):
    """
    Roll forward h steps, feeding predictions back into the lag stack (classic NARX).
    df_last: DataFrame containing the most recent p rows for lag construction
    scaler: (mu, sigma) from training features
    target_name: 'y' or 'pi' for the model passed
    other_targets_updater: optional callback to update other series (e.g., if you jointly roll y, pi, i)
    """
    mu, sigma = scaler
    df_roll = df_last.copy().iloc[-p:].copy()
    preds = []
    for _ in range(horizon):
        # build one-step feature row
        X_parts = []
        for col in lag_cols:
            for L in range(1, p+1):
                X_parts.append(df_roll[col].iloc[-L])
        x = pd.Series(X_parts, index=[f"{c}_lag{L}" for c in lag_cols for L in range(1, p+1)])
        xz = ((x - mu) / sigma).fillna(0.0).values.astype(np.float32)
        xz = torch.tensor(xz).unsqueeze(0)
        yhat = model(xz).item()
        preds.append(yhat)
        # append prediction and advance window
        new_row = df_roll.iloc[-1].copy()
        new_row[target_name] = yhat
        df_roll = pd.concat([df_roll, new_row.to_frame().T], axis=0)
        if other_targets_updater:
            df_roll = other_targets_updater(df_roll)
    return np.array(preds)